Download the file from 

https://raw.githubusercontent.com/Apollo1122/ElasticSearch-Apache-Logs/refs/heads/master/apache_logs

# D05 Exercise — Apache Web Logs

Parse raw Apache access logs with a regular expression, write clean records to CSV, and answer operational questions with pandas.

**Recommended time:** 90 minutes  
**Deliverables:** this completed notebook and `parsed_apache_logs.csv`

## Learning objectives

By the end of this exercise, you should be able to:

- parse semi-structured text using a compiled regex with named groups;
- handle missing values and rejected log lines;
- write dictionaries to CSV safely;
- load, clean, filter, aggregate, and sort data with pandas;
- interpret HTTP 4xx and 5xx errors, response sizes, URLs, IPs, and user agents.

## Apache combined log format

A typical row is:

```text
83.149.9.216 - - [17/May/2015:10:05:03 +0000] "GET /presentations/logstash-monitorama-2013/images/kibana-search.png HTTP/1.1" 200 203023 "http://semicomplete.com/presentations/logstash-monitorama-2013/" "Mozilla/5.0 (...)"
```

The fields are IP address, identity, authenticated user, timestamp, request, status code, content length, referrer, and user agent. In this exercise, retain `ip`, `timestamp`, `method`, `url`, `http_version`, `status_code`, `content_length`, `referrer`, and `user_agent`. A content length of `-` means that its value is unavailable.

## 1. Load and inspect the raw logs — 5 minutes

In [ ]:
from pathlib import Path
import csv
import re

# store this into data directory in C drive
# change this path as per your file location
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'apache_web_logs.txt').exists():
    DATA_DIR = Path.cwd() / 'D05_NumPyPandas'

LOG_PATH = DATA_DIR / 'apache_web_logs.txt'
CSV_PATH = DATA_DIR / 'parsed_apache_logs.csv'

with LOG_PATH.open(encoding='utf-8') as log_file:
    log_lines = [line.rstrip('\n') for line in log_file if line.strip()]

print(f'Loaded {len(log_lines):,} lines from {LOG_PATH}')
print(log_lines[0])

## 2. Build the regular expression — 20 minutes

Complete one regex using **named capture groups**. Useful building blocks include `\S+`, `[^\]]+`, `[^\"]*`, `\d{3}`, and `\d+|-`. Keep the expression anchored with `^` and `$`.

In [ ]:
LOG_PATTERN = re.compile(
    r'^'
    r'(?P<ip>TODO)'
    r' TODO '
    r'\[(?P<timestamp>TODO)\] '
    r'"(?P<method>TODO) (?P<url>TODO) HTTP/(?P<http_version>TODO)" '
    r'(?P<status_code>TODO) '
    r'(?P<content_length>TODO) '
    r'"(?P<referrer>TODO)" '
    r'"(?P<user_agent>TODO)"'
    r'$'
)

Test the pattern on one row before parsing the full file. The cell should run without an assertion error.

In [ ]:
sample_match = LOG_PATTERN.fullmatch(log_lines[0])
assert sample_match is not None, 'The pattern did not match the sample row'
sample_record = sample_match.groupdict()
assert sample_record['ip'] == '83.149.9.216'
assert sample_record['method'] == 'GET'
assert sample_record['status_code'] == '200'
assert sample_record['content_length'] == '203023'
sample_record

## 3. Parse every row — 15 minutes

Create `parsed_records` and `rejected_lines`. Store each rejected item as `(line_number, line)` so it can be investigated. Do not silently discard rows that fail to match. Convert `status_code` to `int`. Convert a numeric `content_length` to `int`, and convert `-` to `None`.

In [ ]:
parsed_records = []
rejected_lines = []

for line_number, line in enumerate(log_lines, start=1):
    # TODO: match the row, transform its values, and append it to the correct list
    pass

print(f'Parsed: {len(parsed_records):,}')
print(f'Rejected: {len(rejected_lines):,}')

In [ ]:
assert len(parsed_records) + len(rejected_lines) == len(log_lines)
assert parsed_records, 'No records were parsed'
assert isinstance(parsed_records[0]['status_code'], int)
assert all(r['content_length'] is None or isinstance(r['content_length'], int) for r in parsed_records)

# The source intentionally contains 14 malformed rows. Inspect them before continuing.
assert len(rejected_lines) == 14, f'Expected 14 malformed rows; first three: {rejected_lines[:3]}'
rejected_lines[:5]

## 4. Store parsed data as CSV — 10 minutes

Use `csv.DictWriter`. Write a header, then all records. Use `newline=''` when opening the file so the code behaves correctly on Windows.

In [ ]:
FIELD_NAMES = [
    'ip', 'timestamp', 'method', 'url', 'http_version',
    'status_code', 'content_length', 'referrer', 'user_agent'
]

# TODO: write parsed_records to CSV_PATH

assert CSV_PATH.exists(), f'{CSV_PATH} was not created'
print(f'CSV written to {CSV_PATH}')

## 5. Load and prepare the CSV with pandas — 10 minutes

In [ ]:
import pandas as pd

logs = pd.read_csv(CSV_PATH)

# TODO: convert timestamp to a timezone-aware datetime
# Hint: format='%d/%b/%Y:%H:%M:%S %z'

# TODO: ensure status_code and content_length are numeric

logs.info()
logs.head()

In [ ]:
assert len(logs) == len(parsed_records)
assert pd.api.types.is_datetime64_any_dtype(logs['timestamp'])
assert pd.api.types.is_numeric_dtype(logs['status_code'])
assert pd.api.types.is_numeric_dtype(logs['content_length'])

## 6. Pandas analytics — 25 minutes

For every question, write pandas code and add a one- or two-sentence interpretation beneath the result.

1. How many total requests are present? How many unique IP addresses and URLs?
2. Create `client_errors` containing all **4xx** requests. Report the count, percentage of all requests, and five most common 4xx URLs.
3. Create `server_errors` containing all **5xx** requests. Report the count, percentage, and status-code breakdown. Your code must still work if there are no 5xx rows.
4. Find the total content delivered in bytes and MiB. Remember that missing lengths should not break the calculation.
5. Show request counts for every status code and HTTP method.
6. Find the ten most requested URLs and the ten IP addresses making the most requests.
7. Find the ten most common user-agent strings. How many rows have a missing user agent represented by `-`?

In [ ]:
# Question 1: overall traffic
# TODO

In [ ]:
# Questions 2 and 3: client and server errors
# TODO

In [ ]:
# Question 4: content volume
# TODO

In [ ]:
# Questions 5 and 6: status codes, methods, URLs, and IPs
# TODO

In [ ]:
# Question 7: user agents
# TODO

## Submission checklist

- All cells run from top to bottom without errors.
- The regex uses named groups and parses all valid rows.
- Missing content lengths are handled as missing values, not zero during parsing.
- `parsed_apache_logs.csv` has a header and one row per parsed log line.
- Analytics use pandas operations rather than manual loops.
- Each analytics result includes a short interpretation.
- The notebook and generated CSV are submitted together.